# Act 4 & 5 · Semantic Property Graph (`GRAPH_EXPAND`), Intersection Consent & Measured Cost Model

Resolving records into a `person_id` is only the halfway mark of a Composable CDP. To activate customer data safely and profitably, we need:
1. **Declarative Semantic Property Graph (`cdp.cdp_semantic_graph`):** Using BigQuery's native `MEASURE()` DDL (`70_graph.sql`) and `GRAPH_EXPAND` + `AGG()`, analysts and **BigQuery Conversational Analytics Data Agents** can aggregate multi-hop metrics (`Profile -> Membership -> Identifier`) without risking fan-out double-counting.
2. **Intersection Consent under the Australian Privacy Act 1988 (`85_consent.sql`):** Under APP 7, an explicit `WITHDRAWN` opt-out on any source record for a `(person, channel, purpose)` must override a `GRANTED` opt-in on another record for that same person.
3. **Downstream Activation & Household Mail De-Duplication (`90_downstream.sql`):** Joining loyalty spend across all resolved records per customer and eliminating duplicate postal mailings within the same household (`cdp.golden_household`).
4. **Measured Cost Model (`96_cost_model.sql`):** Auditing actual token consumption and BigQuery compute from `INFORMATION_SCHEMA.JOBS` and extrapolating to 15M records.

In [1]:
%load_ext bigquery_magics

import bigquery_magics
import pandas as pd
from IPython.display import display
from pathlib import Path

# ---------------------------------------------------------------------------
# Set PROJECT if running outside a checkout that has demo/config.env
PROJECT = ""          # e.g. "all-things-cdp"
DATASET = ""          # leave blank for the default, "cdp"
# ---------------------------------------------------------------------------

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 52)

CFG, CONFIG, _source = {}, None, None

def _find_config():
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        for candidate in (base / "config.env", base / "demo" / "config.env"):
            if candidate.is_file():
                return candidate
    return None

if not PROJECT:
    CONFIG = _find_config()
    if CONFIG:
        for _line in CONFIG.read_text().splitlines():
            _line = _line.strip()
            if _line and not _line.startswith("#") and "=" in _line:
                _k, _v = _line.split("=", 1)
                CFG[_k.strip()] = _v.split("#", 1)[0].strip().strip('"').strip("'")
        PROJECT = CFG.get("CDP_PROJECT", "")
        _source = str(CONFIG)

if not PROJECT:
    import os
    PROJECT = os.environ.get("GOOGLE_CLOUD_PROJECT") or os.environ.get("GCP_PROJECT") or "all-things-cdp"
    _source = "environment / default"

DATASET = DATASET or CFG.get("CDP_DS") or CFG.get("CDP_DATASET_PREFIX") or "cdp"
LOCATION = CFG.get("CDP_LOCATION", "US")

bigquery_magics.context.project = PROJECT
bigquery_magics.context.location = LOCATION
bigquery_magics.context.progress_bar_type = None

print(f"project    {PROJECT}   (from {_source})")
print(f"dataset    {DATASET}   (location {LOCATION})")

project    all-things-cdp   (from demo/config.env)
dataset    cdp   (location US)

## 1 · Declarative Graph Measures with `GRAPH_EXPAND` + `AGG()` (`cdp.cdp_semantic_graph`)

In traditional SQL, joining `Profile (20,000)` $\rightarrow$ `Membership (28,906)` $\rightarrow$ `Identifier (18,724)` fans out profile-level metrics by the number of identifiers per profile.
In `70_graph.sql`, `cdp.cdp_semantic_graph` declares node-scoped measures (`MEASURE(COUNT(1)) AS profile_count`, `MEASURE(COUNT(DISTINCT baseline_wesid)) AS baseline_wesid_count`, etc.). When queried via `FROM GRAPH_EXPAND("cdp.cdp_semantic_graph")`, BigQuery's `AGG()` operator automatically evaluates each measure at its declared grain!

In [2]:
%%bigquery df_semantic_graph
SELECT
  Profile_sources AS brand,
  AGG(Profile_profile_count)                    AS profiles,
  AGG(Membership_link_count)                    AS links,
  AGG(Identifier_identifier_count)              AS distinct_identifiers,
  AGG(Profile_profiles_touching_hubs)           AS touching_hubs,
  AGG(Profile_baseline_wesid_count)             AS baseline_clusters,
  AGG(Profile_weighted_wesid_count)             AS weighted_clusters,
  AGG(Profile_composable_wesid_count)           AS composable_clusters,
  AGG(Profile_profiles_in_baseline_hairballs)   AS in_baseline_hairballs,
  AGG(Profile_profiles_in_weighted_hairballs)   AS in_weighted_hairballs,
  AGG(Profile_profiles_in_composable_hairballs) AS in_composable_hairballs
FROM GRAPH_EXPAND("cdp.cdp_semantic_graph")
GROUP BY brand
ORDER BY profiles DESC

Fan-Out-Free Multi-Hop Aggregation via GRAPH_EXPAND + AGG() (cdp.cdp_semantic_graph)
brand    profiles  links  distinct_ids  touching_hubs  baseline_clusters  weighted_clusters  composable_clusters  baseline_hairball  composable_hairball
-------  --------  -----  ------------  -------------  -----------------  -----------------  -------------------  -----------------  -------------------
POS      5593      2190   2117          505            3918               5340               5192                 650                0                  
CRM      4155      8451   7834          381            2002               4125               4125                 1455               0                  
ECOM     3752      5573   5268          355            1986               3716               3719                 1185               0                  
LOYALTY  3248      6674   6235          307            1732               3233               3232                 1087               0                

brand,profiles,links,distinct_ids,touching_hubs,baseline_clusters,weighted_clusters,composable_clusters,baseline_hairball,composable_hairball
POS,5593,2190,2117,505,3918,5340,5192,650,0
CRM,4155,8451,7834,381,2002,4125,4125,1455,0
ECOM,3752,5573,5268,355,1986,3716,3719,1185,0
LOYALTY,3248,6674,6235,307,1732,3233,3232,1087,0
ENRICH,1842,1945,1891,189,1075,1833,1835,526,0
SUPPORT,968,1068,1039,98,662,962,965,224,0
CALL,442,329,321,56,325,442,442,68,0


## 2 · Intersection Consent Governance (`cdp.v_consent_impact` & `cdp.v_consent_conflicts`)

When a customer has 3 source records — two saying `GRANTED` for marketing email and one saying `WITHDRAWN` — a naive **Union Rule** (`ANY(GRANTED)`) illegally contacts a customer who explicitly opted out!
Stage 85 (`85_consent.sql`) enforces **Intersection Consent**: a single `WITHDRAWN` assertion across any resolved record in the cluster suppresses marketing on that channel (`actual_decision = 'WITHDRAWN'`), and any person carrying a `RISK_FLAG` is automatically suppressed across all marketing channels.

In [3]:
%%bigquery df_consent
SELECT
  channel,
  purpose,
  people_with_a_stated_preference,
  audience_union_rule,
  audience_intersection_rule,
  contacts_the_union_rule_would_add,
  of_which_explicitly_withdrawn,
  suppressed_for_risk,
  pct_of_union_audience_unlawful
FROM `cdp.v_consent_impact`
WHERE purpose = 'MARKETING'
ORDER BY people_with_a_stated_preference DESC

Privacy Act 1988 Compliance: Union vs Intersection Consent (cdp.v_consent_impact)
channel  purpose    stated_pref  union_rule  intersection_rule  union_adds  explicitly_withdrawn  risk_suppressed  pct_unlawful
-------  ---------  -----------  ----------  -----------------  ----------  --------------------  ---------------  ------------
EMAIL    MARKETING  3370         2070        1876               194         71                    66               9.37        
SMS      MARKETING  2276         1379        1290               89          30                    66               6.45        
POST     MARKETING  1701         1001        954                47          17                    66               4.7         
PHONE    MARKETING  1273         757         734                23          6                     66               3.04        

channel,purpose,stated_pref,union_rule,intersection_rule,union_adds,explicitly_withdrawn,risk_suppressed,pct_unlawful
EMAIL,MARKETING,3370,2070,1876,194,71,66,9.37
SMS,MARKETING,2276,1379,1290,89,30,66,6.45
POST,MARKETING,1701,1001,954,47,17,66,4.7
PHONE,MARKETING,1273,757,734,23,6,66,3.04


In [4]:
%%bigquery df_consent_conflicts
SELECT
  person_id,
  full_name,
  channel,
  granted_count,
  withdrawn_count,
  would_have_been_contacted,
  actual_decision,
  explanation
FROM `cdp.v_consent_conflicts`
LIMIT 5

Customers Protected from Unlawful Contact by Intersection Consent (cdp.v_consent_conflicts)
person_id             full_name       channel  granted  withdrawn  union_would_do  actual_decision  explanation                                     
--------------------  --------------  -------  -------  ---------  --------------  ---------------  ------------------------------------------------
PER-4490c5b70d037280  Lucille Osman   POST     2        1          GRANTED         WITHDRAWN        Withdrawn on 2019-05-21. Withdrawal is absolu...
PER-52c80bb634a27ec5  Tengfei Chi     EMAIL    2        1          GRANTED         WITHDRAWN        Withdrawn on 2023-10-19. Withdrawal is absolu...
PER-79474a4aaffe6762  Malti Nagpal    EMAIL    2        1          GRANTED         WITHDRAWN        Withdrawn on 2021-07-15. Withdrawal is absolu...
PER-0077751c2ba5e056  Salil Kyriazis  EMAIL    1        1          GRANTED         WITHDRAWN        Withdrawn on 2021-07-25. Withdrawal is absolu...
PER-5e76850a2e

person_id,full_name,channel,granted,withdrawn,union_would_do,actual_decision,explanation
PER-4490c5b70d037280,Lucille Osman,POST,2,1,GRANTED,WITHDRAWN,Withdrawn on 2019-05-21. Withdrawal is absolute and is not overridden by a later grant from another source.
PER-52c80bb634a27ec5,Tengfei Chi,EMAIL,2,1,GRANTED,WITHDRAWN,Withdrawn on 2023-10-19. Withdrawal is absolute and is not overridden by a later grant from another source.
PER-79474a4aaffe6762,Malti Nagpal,EMAIL,2,1,GRANTED,WITHDRAWN,Withdrawn on 2021-07-15. Withdrawal is absolute and is not overridden by a later grant from another source.
PER-0077751c2ba5e056,Salil Kyriazis,EMAIL,1,1,GRANTED,WITHDRAWN,Withdrawn on 2021-07-25. Withdrawal is absolute and is not overridden by a later grant from another source.
PER-5e76850a2e7e8d7d,Austn Maddox,POST,1,1,GRANTED,WITHDRAWN,Withdrawn on 2025-07-07. Withdrawal is absolute and is not overridden by a later grant from another source.


## 3 · Downstream Activation Lift & Household Direct-Mail Savings (`90_downstream.sql`)

Let's inspect the two commercial ROI views built in Stage 90:
1. **`cdp.v_activation_before_after`:** Connecting POS and ECOM transactions across all resolved records increases attributed customer reach from **1,658 loyalty cards (`$162,366` spend)** to **5,316 resolved customers (`$479,989` spend)** — nearly **3x attributed revenue**.
2. **`cdp.v_household_waste`:** Grouping resolved people by `golden_household` (`address_key`) eliminates **13.33% of postal direct mailings** (`115` duplicate catalogue mailings avoided across `748` mailable households, including `31` multi-surname households).

In [5]:
%%bigquery df_roi
SELECT * FROM `cdp.v_activation_before_after`

Customer Spend Attribution Before vs After Identity Resolution (cdp.v_activation_before_after)
basis                                         customers  total_spend  spend_per_customer
--------------------------------------------  ---------  -----------  ------------------
resolved · one person = one customer          5316       479989.09    90.29             
unresolved · one loyalty card = one customer  1658       162366.00    97.93             

basis,customers,total_spend,spend_per_customer
resolved · one person = one customer,5316,479989.09,90.29
unresolved · one loyalty card = one customer,1658,162366.00,97.93


In [6]:
%%bigquery df_waste
SELECT * FROM `cdp.v_household_waste`

Direct-Mail Catalogue Waste Eliminated by Household Resolution (cdp.v_household_waste)
households  people  duplicate_mailings_avoided  pct_of_mailings_avoidable  multi_person_households  multi_surname_households
----------  ------  --------------------------  -------------------------  -----------------------  ------------------------
748         863     115                         13.33                      104                      31                      

households,people,duplicate_mailings_avoided,pct_of_mailings_avoidable,multi_person_households,multi_surname_households
748,863,115,13.33,104,31


## 4 · Measured AI & Compute Cost Model (`cdp.v_cost_model` & `cdp.v_cost_compute`)

Because 99.61% of candidate pairs are settled deterministically by the 2-Hop Graph + Hybrid Search tiering rules, the **entire 20,000-record AI pipeline costs less than `$0.09` in Gemini/embedding calls** (`$0.0043` per 1,000 records, or **`~$63.80` per full rebuild at 15M records**).

In [7]:
%%bigquery df_cost
SELECT
  demo_records,
  demo_people_resolved,
  ai_calls,
  total_tokens,
  usd_extraction,
  usd_embedding,
  usd_adjudication,
  usd_model_total,
  usd_model_per_1k_records,
  target_records,
  usd_model_at_target_per_rebuild
FROM `cdp.v_cost_model`

Measured AI Token & Model Cost Summary (cdp.v_cost_model)
demo_records  resolved_people  ai_calls  total_tokens  usd_extract  usd_embed  usd_model_total  usd_per_1k  target_records  usd_at_15m_rebuild
------------  ---------------  --------  ------------  -----------  ---------  ---------------  ----------  --------------  ------------------
20000         14016            105883    409125        0.0367       0.0483     0.0851           0.0043      15000000        63.8              

demo_records,resolved_people,ai_calls,total_tokens,usd_extract,usd_embed,usd_model_total,usd_per_1k,target_records,usd_at_15m_rebuild
20000,14016,105883,409125,0.0367,0.0483,0.0851,0.0043,15000000,63.8
